# Day 3: RAG Optimization - Reranking & Query Enhancement

In this session, we will:
1. Implement query rewriting for better retrieval.
2. Add a reranker to improve result quality.
3. Evaluate RAG performance using LLM-as-a-Judge.

## Prerequisites
```bash
pip install litellm chromadb sentence-transformers
```

!pip install litellm chromadb sentence-transformers

In [1]:
!pip install litellm chromadb sentence-transformers

  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.3 MB/s  0:00:08m0:00:0100:01m
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 3.5 MB/s  0:00:22m0:00:0100:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]


In [1]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import numpy as np
from litellm import completion, embedding


/Users/rajesh/miniconda3/envs/rag_session/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup ChromaDB (Reusing Day 2 Data)

We'll use the same knowledge base from Day 2.

In [2]:
# Initialize ChromaDB
client = chromadb.Client(Settings(
    persist_directory="/Users/rajesh/Desktop/rajesh/Archive/teaching/RAG_sessions/Day-2/chroma_db"
))


collection_name = "rag_workshop_day3"


# Create fresh collection
try:
    client.delete_collection(name=collection_name)
except:
    pass

collection = client.create_collection(name=collection_name)

# Sample documents
documents = [
    "Retrieval-Augmented Generation (RAG) combines retrieval and generation to reduce hallucinations.",
    "Vector databases like ChromaDB enable fast similarity search using embeddings.",
    "Reranking improves retrieval quality by re-scoring results with cross-encoders.",
    "Query rewriting transforms user queries into better search queries.",
    "Hybrid search combines dense embeddings with sparse keyword matching.",
    "LiteLM provides a unified interface for calling multiple LLM providers.",
    "Chunking strategies include fixed-size, recursive, and semantic approaches.",
    "Context precision measures the relevance of retrieved chunks.",
]

def get_embeddings(texts, model="ollama/nomic-embed-text"):
    response = embedding(model=model, input=texts)
    return [item['embedding'] for item in response['data']]

embeddings = get_embeddings(documents)
collection.add(
    embeddings=embeddings,
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print(f"Indexed {len(documents)} documents")





Indexed 8 documents


## 2. Query Rewriting

Transform vague queries into more specific ones.

In [6]:
def rewrite_query(original_query, llm_model="ollama/llama3.2:1b"):
    """Use an LLM to rewrite the query for better retrieval"""
    prompt = f"""You are a query optimization assistant. Rewrite the following query to be more specific and suitable for semantic search. Only return the rewritten query, nothing else.

Original Query: {original_query}

Rewritten Query:"""
    
    response = completion(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response['choices'][0]['message']['content'].strip()





original = "How to make it better?"
rewritten = rewrite_query(original)

In [7]:
print(f"Original: {original}")
print(f"Rewritten: {rewritten}")

Original: How to make it better?
Rewritten: ```sql
SELECT 
    CASE 
        WHEN COUNT(*) > 10 THEN 'better' 
        ELSE 'not good enough'
    END AS recommendation
FROM table_name;
```


## 3. Reranking with Cross-Encoder

Cross-encoders are more accurate than bi-encoders (embeddings) but slower.

In [ ]:
# Load cross-encoder model
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1175.48it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
def retrieve_and_rerank(query, initial_k=10, final_k=3):
    """Retrieve more results, then rerank to get the best ones"""
    # Step 1: Initial retrieval (fast, approximate)
    query_embedding = get_embeddings([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(initial_k, collection.count())
    )
    
    documents = results['documents'][0]
    
    if not documents:
        return []
    
    # Step 2: Rerank (slow, accurate)
    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)
    
    # Step 3: Sort by reranker scores
    ranked_results = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return ranked_results[:final_k]

In [13]:
query = "What improves search quality?"
results = retrieve_and_rerank(query, initial_k=5, final_k=3)




print(f"Query: {query}\n")
print("Reranked Results:")
for i, (doc, score) in enumerate(results):
    print(f"{i+1}. (Score: {score:.4f}) {doc}")

Query: What improves search quality?

Reranked Results:
1. (Score: 1.8577) Reranking improves retrieval quality by re-scoring results with cross-encoders.
2. (Score: -1.2705) Query rewriting transforms user queries into better search queries.
3. (Score: -8.6548) Hybrid search combines dense embeddings with sparse keyword matching.


In [12]:
results

[('Reranking improves retrieval quality by re-scoring results with cross-encoders.',
  np.float32(1.8577018)),
 ('Query rewriting transforms user queries into better search queries.',
  np.float32(-1.2705455)),
 ('Hybrid search combines dense embeddings with sparse keyword matching.',
  np.float32(-8.654774))]

## 4. Complete Optimized RAG Pipeline

In [16]:
def optimized_rag(question, use_rewrite=True, use_rerank=True, llm_model="ollama/llama3.2:1b"):
    """RAG with query rewriting and reranking"""
    print(f"Original Question: {question}\n")
    
    # Step 1: Query Rewriting (optional)
    if use_rewrite:
        query = rewrite_query(question, llm_model)
        print(f"Rewritten Query: {query}\n")
    else:
        query = question
    
    # Step 2: Retrieve and Rerank (optional)
    if use_rerank:
        results = retrieve_and_rerank(query, initial_k=8, final_k=3)
        context_chunks = [doc for doc, _ in results]
    else:
        query_embedding = get_embeddings([query])[0]
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=3
        )
        context_chunks = results['documents'][0]
    
    print("Retrieved Context:")
    for i, chunk in enumerate(context_chunks):
        print(f"{i+1}. {chunk}")
    
    # Step 3: Generate Answer
    context = "\n\n".join(context_chunks)
    prompt = f"""Answer the question based on the context. Be concise.

Context:
{context}

Question: {question}

Answer:"""
    
    response = completion(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response['choices'][0]['message']['content']
    print(f"\nAnswer:\n{answer}")
    return answer


In [17]:
# Compare with and without optimization
print("=" * 80)
print("WITHOUT OPTIMIZATION")
print("=" * 80)
optimized_rag("How to improve results?", use_rewrite=False, use_rerank=False)


WITHOUT OPTIMIZATION
Original Question: How to improve results?

Retrieved Context:
1. Reranking improves retrieval quality by re-scoring results with cross-encoders.
2. Retrieval-Augmented Generation (RAG) combines retrieval and generation to reduce hallucinations.
3. Query rewriting transforms user queries into better search queries.

Answer:
To improve results, you can consider the following strategies:

1. **Use robust retrieval models**: Invest in high-quality retrieval algorithms that can accurately match query intent with document content.
2. **Optimize search engine architecture**: Ensure your search engine's architecture is designed to handle diverse user queries and reduce hallucinations.
3. **Improve natural language understanding**: Enhance your language model's ability to comprehend user queries and generate relevant responses.

These strategies can help improve the quality of results retrieved by the system.


"To improve results, you can consider the following strategies:\n\n1. **Use robust retrieval models**: Invest in high-quality retrieval algorithms that can accurately match query intent with document content.\n2. **Optimize search engine architecture**: Ensure your search engine's architecture is designed to handle diverse user queries and reduce hallucinations.\n3. **Improve natural language understanding**: Enhance your language model's ability to comprehend user queries and generate relevant responses.\n\nThese strategies can help improve the quality of results retrieved by the system."

In [18]:
print("=" * 80)
print("WITH OPTIMIZATION")
print("=" * 80)
optimized_rag("How to improve results?", use_rewrite=True, use_rerank=True)

WITH OPTIMIZATION
Original Question: How to improve results?

Rewritten Query: "Optimize database queries for fast and accurate information retrieval."

Retrieved Context:
1. Vector databases like ChromaDB enable fast similarity search using embeddings.
2. Reranking improves retrieval quality by re-scoring results with cross-encoders.
3. Query rewriting transforms user queries into better search queries.

Answer:
To improve results in vector databases and reranking, you can try the following strategies:

1. **Optimize query vectors**: Use techniques like dimensionality reduction or perturbation to make query vectors more relevant.
2. **Use robust ranking methods**: Implement methods like cosine similarity, Jaccard similarity, or modified Euclidean distance that are less affected by noise and outliers.
3. **Apply re-ranking algorithms**: Utilize pre-trained cross-encoders like SimRank, Jaccard similarity, or L1/mean normalization to re-score results.
4. **Rewrite queries for better sema

'To improve results in vector databases and reranking, you can try the following strategies:\n\n1. **Optimize query vectors**: Use techniques like dimensionality reduction or perturbation to make query vectors more relevant.\n2. **Use robust ranking methods**: Implement methods like cosine similarity, Jaccard similarity, or modified Euclidean distance that are less affected by noise and outliers.\n3. **Apply re-ranking algorithms**: Utilize pre-trained cross-encoders like SimRank, Jaccard similarity, or L1/mean normalization to re-score results.\n4. **Rewrite queries for better semantics**: Use query rewriting techniques like part-of-speech tagging, named entity recognition, or semantic parsing to transform user queries into more effective search queries.'

## 5. LLM-as-a-Judge Evaluation

Use an LLM to evaluate the quality of RAG responses.

In [19]:
def evaluate_rag_response(question, answer, context, llm_model="ollama/gemma3:4b"):
    """Evaluate RAG response for faithfulness and relevance"""
    eval_prompt = f"""You are an evaluation assistant. Evaluate the following RAG system response.

Question: {question}

Context:
{context}

Answer:
{answer}

Evaluate on a scale of 1-5:
1. Faithfulness: Is the answer grounded in the context? (1=not at all, 5=completely)
2. Relevance: Does the answer address the question? (1=not at all, 5=completely)

Respond in this format:
Faithfulness: [score]
Relevance: [score]
Explanation: [brief explanation]"""
    
    response = completion(
        model=llm_model,
        messages=[{"role": "user", "content": eval_prompt}]
    )
    
    return response['choices'][0]['message']['content']

# Test evaluation
test_question = "What is reranking?"
test_context = "Reranking improves retrieval quality by re-scoring results with cross-encoders."
test_answer = "Reranking is a technique that uses cross-encoders to re-score search results and improve quality."

evaluation = evaluate_rag_response(test_question, test_answer, test_context)
print("Evaluation Result:")
print(evaluation)

Evaluation Result:
Faithfulness: 5
Relevance: 5
Explanation: The answer is entirely faithful to the provided context, directly stating that reranking uses cross-encoders to re-score results. It also perfectly addresses the question "What is reranking?" by defining it succinctly.
